# Temporal OOD Experiments

**Goal**: Evaluate CAGP under temporal distribution shift.

**Setup**: Train on older triples, test on newer triples.

**Dataset**: ICEWS14 (Integrated Crisis Early Warning System)
- 7,128 entities
- 230 relations  
- 90,730 triples with timestamps (2014-01-01 to 2014-12-31)

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict
from sklearn.metrics import roc_auc_score
import json
import os

## 1. Download and Load ICEWS14

In [ ]:
# Download ICEWS14 if not present
DATA_DIR = '../data/raw/icews14'
os.makedirs(DATA_DIR, exist_ok=True)

# ICEWS14 format: head, relation, tail, timestamp
# Standard splits available from various sources
# We'll use the tkbc library format

ICEWS14_URL = "https://github.com/facebookresearch/tkbc/raw/main/data/ICEWS14/"

import urllib.request

files = ['train.txt', 'valid.txt', 'test.txt']
for f in files:
    fpath = os.path.join(DATA_DIR, f)
    if not os.path.exists(fpath):
        print(f"Downloading {f}...")
        urllib.request.urlretrieve(f"{ICEWS14_URL}{f}", fpath)
        print(f"Downloaded {f}")

In [ ]:
def load_icews14(data_dir):
    """Load ICEWS14 with timestamps."""
    triples = []
    
    for split in ['train', 'valid', 'test']:
        fpath = os.path.join(data_dir, f'{split}.txt')
        with open(fpath) as f:
            for line in f:
                parts = line.strip().split('\t')
                h, r, t, timestamp = parts[0], parts[1], parts[2], parts[3]
                # Parse timestamp (format varies: could be date or day index)
                triples.append({
                    'head': h,
                    'relation': r,
                    'tail': t,
                    'timestamp': timestamp,
                    'split': split
                })
    
    df = pd.DataFrame(triples)
    
    # Build entity and relation maps
    entities = sorted(set(df['head'].tolist() + df['tail'].tolist()))
    relations = sorted(set(df['relation'].tolist()))
    
    entity2id = {e: i for i, e in enumerate(entities)}
    relation2id = {r: i for i, r in enumerate(relations)}
    
    # Convert to IDs
    df['head_id'] = df['head'].map(entity2id)
    df['relation_id'] = df['relation'].map(relation2id)
    df['tail_id'] = df['tail'].map(entity2id)
    
    return df, entity2id, relation2id

df, entity2id, relation2id = load_icews14(DATA_DIR)
print(f"Entities: {len(entity2id)}")
print(f"Relations: {len(relation2id)}")
print(f"Triples: {len(df)}")
print(f"\nSample:\n{df.head()}")

## 2. Create Temporal Split

**Protocol**:
- Train: First 6 months (Jan-Jun 2014)
- Test ID: Last 6 months, entities seen in training
- Test OOD: Last 6 months, NEW entities OR entity-relation pairs

In [ ]:
def parse_timestamp(ts):
    """Parse ICEWS timestamp to day index."""
    # ICEWS14 timestamps are typically day indices (0-364) or dates
    try:
        return int(ts)  # Day index
    except ValueError:
        # Try parsing as date
        dt = datetime.strptime(ts, '%Y-%m-%d')
        return (dt - datetime(2014, 1, 1)).days

df['day'] = df['timestamp'].apply(parse_timestamp)
print(f"Day range: {df['day'].min()} to {df['day'].max()}")

In [ ]:
def temporal_split(df, cutoff_day=182):  # Day 182 = July 1
    """Split by time, then categorize test triples."""
    
    train = df[df['day'] < cutoff_day].copy()
    future = df[df['day'] >= cutoff_day].copy()
    
    # Entities seen in training
    train_entities = set(train['head_id'].tolist() + train['tail_id'].tolist())
    
    # Entity-relation pairs seen in training
    train_pairs = set()
    for _, row in train.iterrows():
        train_pairs.add((row['head_id'], row['relation_id']))
        train_pairs.add((row['tail_id'], row['relation_id']))
    
    # Categorize future triples
    def categorize(row):
        h_seen = row['head_id'] in train_entities
        t_seen = row['tail_id'] in train_entities
        h_pair = (row['head_id'], row['relation_id']) in train_pairs
        t_pair = (row['tail_id'], row['relation_id']) in train_pairs
        
        if not h_seen or not t_seen:
            return 'new_entity'  # OOD: unseen entity
        elif not h_pair or not t_pair:
            return 'new_pair'    # OOD: new entity-relation combination
        else:
            return 'id'          # ID: all components seen
    
    future['category'] = future.apply(categorize, axis=1)
    
    return train, future

train_df, future_df = temporal_split(df)

print(f"Training triples: {len(train_df)}")
print(f"\nFuture triple categories:")
print(future_df['category'].value_counts())

## 3. Build Coverage Matrix

In [ ]:
def build_coverage_matrix(train_df, n_entities, n_relations):
    """Build coverage matrix from training data."""
    coverage = np.zeros((n_entities, n_relations), dtype=np.float32)
    
    for _, row in train_df.iterrows():
        coverage[row['head_id'], row['relation_id']] = 1
        coverage[row['tail_id'], row['relation_id']] = 1
    
    return coverage

n_entities = len(entity2id)
n_relations = len(relation2id)

coverage = build_coverage_matrix(train_df, n_entities, n_relations)
print(f"Coverage matrix: {coverage.shape}")
print(f"Coverage density: {coverage.mean():.4f}")

## 4. Implement GP-KGE for Temporal Data

In [ ]:
from src.models.gp_kge import GPKGE
from src.models.coverage_augmented_gpkge import CoverageAugmentedGPKGE

# If imports fail, define inline
try:
    from src.models.gp_kge import GPKGE
except ImportError:
    print("Defining models inline...")
    
    class GPKGE(torch.nn.Module):
        def __init__(self, n_entities, n_relations, dim=100):
            super().__init__()
            self.entity_mean = torch.nn.Embedding(n_entities, dim)
            self.entity_logvar = torch.nn.Embedding(n_entities, dim)
            self.relation_emb = torch.nn.Embedding(n_relations, dim)
            
            # Initialize
            torch.nn.init.xavier_uniform_(self.entity_mean.weight)
            torch.nn.init.constant_(self.entity_logvar.weight, -2.0)
            torch.nn.init.xavier_uniform_(self.relation_emb.weight)
        
        def forward(self, heads, relations, tails):
            h_mean = self.entity_mean(heads)
            t_mean = self.entity_mean(tails)
            r = self.relation_emb(relations)
            
            # DistMult scoring
            score = (h_mean * r * t_mean).sum(dim=-1)
            return score
        
        def get_variance(self, entities):
            logvar = self.entity_logvar(entities)
            return torch.exp(logvar).mean(dim=-1)
        
        def get_gp_uncertainty(self, heads, tails):
            h_var = self.get_variance(heads)
            t_var = self.get_variance(tails)
            return (h_var + t_var) / 2

In [ ]:
def train_gpkge(model, train_df, n_entities, epochs=30, batch_size=1024, lr=0.001, device='cuda'):
    """Train GP-KGE on temporal training data."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.BCEWithLogitsLoss()
    
    # Prepare data
    heads = torch.tensor(train_df['head_id'].values)
    relations = torch.tensor(train_df['relation_id'].values)
    tails = torch.tensor(train_df['tail_id'].values)
    
    n_triples = len(heads)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        # Shuffle
        perm = torch.randperm(n_triples)
        
        for i in range(0, n_triples, batch_size):
            idx = perm[i:i+batch_size]
            
            h = heads[idx].to(device)
            r = relations[idx].to(device)
            t = tails[idx].to(device)
            
            # Positive scores
            pos_scores = model(h, r, t)
            
            # Negative sampling (random tails)
            neg_t = torch.randint(0, n_entities, (len(idx),)).to(device)
            neg_scores = model(h, r, neg_t)
            
            # Labels
            pos_labels = torch.ones_like(pos_scores)
            neg_labels = torch.zeros_like(neg_scores)
            
            # Loss
            scores = torch.cat([pos_scores, neg_scores])
            labels = torch.cat([pos_labels, neg_labels])
            loss = criterion(scores, labels)
            
            # KL regularization on variances
            kl = 0.5 * (model.entity_logvar.weight.exp() - model.entity_logvar.weight - 1).mean()
            loss = loss + 0.01 * kl
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")
    
    return model

## 5. Temporal OOD Evaluation

In [ ]:
def evaluate_temporal_ood(model, coverage, future_df, device='cuda'):
    """Evaluate OOD detection on temporal test set."""
    model.eval()
    
    # Separate ID and OOD
    id_df = future_df[future_df['category'] == 'id']
    ood_df = future_df[future_df['category'].isin(['new_entity', 'new_pair'])]
    
    print(f"ID samples: {len(id_df)}")
    print(f"OOD samples: {len(ood_df)} (new_entity: {len(future_df[future_df['category']=='new_entity'])}, new_pair: {len(future_df[future_df['category']=='new_pair'])})")
    
    def get_uncertainties(df):
        heads = torch.tensor(df['head_id'].values).to(device)
        relations = torch.tensor(df['relation_id'].values).to(device)
        tails = torch.tensor(df['tail_id'].values).to(device)
        
        with torch.no_grad():
            # GP uncertainty
            gp_unc = model.get_gp_uncertainty(heads, tails).cpu().numpy()
        
        # Coverage uncertainty
        h_np = heads.cpu().numpy()
        r_np = relations.cpu().numpy()
        t_np = tails.cpu().numpy()
        
        cov_unc = 2 - coverage[h_np, r_np] - coverage[t_np, r_np]
        
        return gp_unc, cov_unc
    
    id_gp, id_cov = get_uncertainties(id_df)
    ood_gp, ood_cov = get_uncertainties(ood_df)
    
    # Compute AUROC
    # Higher uncertainty should indicate OOD
    
    # Labels: 0 = ID, 1 = OOD
    labels = np.concatenate([np.zeros(len(id_gp)), np.ones(len(ood_gp))])
    
    # GP-only AUROC
    gp_scores = np.concatenate([id_gp, ood_gp])
    auroc_gp = roc_auc_score(labels, gp_scores)
    
    # Coverage-only AUROC
    cov_scores = np.concatenate([id_cov, ood_cov])
    auroc_cov = roc_auc_score(labels, cov_scores)
    
    # CAGP (alpha=0.5 for simplicity)
    # Normalize GP to coverage scale
    gp_norm = gp_scores * cov_scores.mean() / (gp_scores.mean() + 1e-8)
    cagp_scores = 0.5 * gp_norm + 0.5 * cov_scores
    auroc_cagp = roc_auc_score(labels, cagp_scores)
    
    return {
        'GP-only': auroc_gp,
        'Coverage-only': auroc_cov,
        'CAGP': auroc_cagp,
        'synergy': auroc_cagp - max(auroc_gp, auroc_cov)
    }

## 6. Run Experiment

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Train model
model = GPKGE(n_entities, n_relations, dim=100)
model = train_gpkge(model, train_df, n_entities, epochs=30, device=device)

In [ ]:
# Evaluate
results = evaluate_temporal_ood(model, coverage, future_df, device=device)

print("\n" + "="*50)
print("TEMPORAL OOD RESULTS (ICEWS14)")
print("="*50)
for method, auroc in results.items():
    print(f"{method}: {auroc:.4f}")

## 7. Breakdown by OOD Type

In [ ]:
def evaluate_by_ood_type(model, coverage, future_df, device='cuda'):
    """Separate evaluation for new_entity vs new_pair OOD."""
    model.eval()
    
    id_df = future_df[future_df['category'] == 'id']
    new_entity_df = future_df[future_df['category'] == 'new_entity']
    new_pair_df = future_df[future_df['category'] == 'new_pair']
    
    def get_uncertainties(df):
        if len(df) == 0:
            return np.array([]), np.array([])
        
        heads = torch.tensor(df['head_id'].values).to(device)
        relations = torch.tensor(df['relation_id'].values).to(device)
        tails = torch.tensor(df['tail_id'].values).to(device)
        
        with torch.no_grad():
            gp_unc = model.get_gp_uncertainty(heads, tails).cpu().numpy()
        
        h_np = heads.cpu().numpy()
        r_np = relations.cpu().numpy()
        t_np = tails.cpu().numpy()
        
        cov_unc = 2 - coverage[h_np, r_np] - coverage[t_np, r_np]
        
        return gp_unc, cov_unc
    
    id_gp, id_cov = get_uncertainties(id_df)
    ne_gp, ne_cov = get_uncertainties(new_entity_df)
    np_gp, np_cov = get_uncertainties(new_pair_df)
    
    results = {}
    
    # ID vs New Entity
    if len(ne_gp) > 0:
        labels = np.concatenate([np.zeros(len(id_gp)), np.ones(len(ne_gp))])
        
        gp_scores = np.concatenate([id_gp, ne_gp])
        cov_scores = np.concatenate([id_cov, ne_cov])
        
        gp_norm = gp_scores * cov_scores.mean() / (gp_scores.mean() + 1e-8)
        cagp_scores = 0.5 * gp_norm + 0.5 * cov_scores
        
        results['new_entity'] = {
            'GP-only': roc_auc_score(labels, gp_scores),
            'Coverage-only': roc_auc_score(labels, cov_scores),
            'CAGP': roc_auc_score(labels, cagp_scores),
            'n_samples': len(ne_gp)
        }
    
    # ID vs New Pair (entity seen, but new relation context)
    if len(np_gp) > 0:
        labels = np.concatenate([np.zeros(len(id_gp)), np.ones(len(np_gp))])
        
        gp_scores = np.concatenate([id_gp, np_gp])
        cov_scores = np.concatenate([id_cov, np_cov])
        
        gp_norm = gp_scores * cov_scores.mean() / (gp_scores.mean() + 1e-8)
        cagp_scores = 0.5 * gp_norm + 0.5 * cov_scores
        
        results['new_pair'] = {
            'GP-only': roc_auc_score(labels, gp_scores),
            'Coverage-only': roc_auc_score(labels, cov_scores),
            'CAGP': roc_auc_score(labels, cagp_scores),
            'n_samples': len(np_gp)
        }
    
    return results

breakdown = evaluate_by_ood_type(model, coverage, future_df, device=device)

print("\n" + "="*50)
print("BREAKDOWN BY OOD TYPE")
print("="*50)
for ood_type, metrics in breakdown.items():
    print(f"\n{ood_type.upper()} (n={metrics['n_samples']}):")
    print(f"  GP-only:      {metrics['GP-only']:.4f}")
    print(f"  Coverage-only: {metrics['Coverage-only']:.4f}")
    print(f"  CAGP:         {metrics['CAGP']:.4f}")

## 8. Save Results

In [ ]:
output = {
    'dataset': 'ICEWS14',
    'split': 'temporal (first 6 months train, last 6 months test)',
    'overall': results,
    'breakdown': breakdown,
    'metadata': {
        'n_entities': n_entities,
        'n_relations': n_relations,
        'train_triples': len(train_df),
        'test_id': len(future_df[future_df['category'] == 'id']),
        'test_ood_new_entity': len(future_df[future_df['category'] == 'new_entity']),
        'test_ood_new_pair': len(future_df[future_df['category'] == 'new_pair']),
    }
}

with open('../outputs/temporal_ood_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Results saved to outputs/temporal_ood_results.json")

## 9. Key Insights

**Expected Results**:
1. **New Entity OOD**: GP should excel (high variance for unseen entities)
2. **New Pair OOD**: Coverage should excel (entity seen, but new relation context)
3. **CAGP**: Should capture both failure modes, achieving highest overall AUROC

**Why Temporal OOD Matters**:
- Real-world KGs evolve: new entities appear, relations expand
- Random corruption doesn't capture this temporal structure
- If CAGP excels here, it validates robustness to realistic distribution shifts